# 03 - Run allocation

This notebook combines prepared demand/context data with prepared workforce capacity.
It creates the final MSOA/month/bucket allocation output.

In [ ]:
import pandas as pd
from pathlib import Path
import sqlite3

## Settings

`RHO` controls the split between baseline and demand-based allocation.
For the final setup it is set to 50/50.

Greater Manchester uses baseline-only allocation because its crime demand data is known to be unreliable.

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in ["notebooks", "allocation"]:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ALLOC_DB_PATH = DATA_DIR / "allocation_model.db"

RHO = 0.5
RURALITY_WEIGHT = 0.1

# Greater Manchester demand data is known to be unreliable, so the model uses
# baseline-only allocation for the full force capacity there.
BASELINE_ONLY_PFA_CODES = ["E23000005"]

UNCERTAINTY_IQR_RATIO_THRESHOLD = 1.25
UNCERTAINTY_CHANGE_PCT_THRESHOLD = 10
UNCERTAINTY_CHANGE_ABS_THRESHOLD = 2

EQUAL_BUCKET_SHARE = 1 / 3

bucket_baseline_shares = pd.DataFrame(
    [
        {"bucket": "local_reassurance", "bucket_baseline_share": EQUAL_BUCKET_SHARE},
        {"bucket": "acquisitive_crime", "bucket_baseline_share": EQUAL_BUCKET_SHARE},
        {"bucket": "disorder_damage", "bucket_baseline_share": EQUAL_BUCKET_SHARE},
    ]
)

## Load prepared tables

In [ ]:
with sqlite3.connect(ALLOC_DB_PATH) as conn:
    msoa_bucket_demand = pd.read_sql_query("SELECT * FROM msoa_bucket_demand;", conn)
    msoa_context = pd.read_sql_query("SELECT * FROM msoa_context;", conn)
    force_capacity_model = pd.read_sql_query("SELECT * FROM force_capacity;", conn)

capacity_source = "local_policing"

print("demand rows:", len(msoa_bucket_demand))
print("MSOA rows:", len(msoa_context))
print("capacity rows:", len(force_capacity_model))

## Baseline allocation

Baseline allocation depends on population, with a small rurality uplift.

In [ ]:
msoa_context["baseline_score"] = (
    msoa_context["population"]
    * (1 + RURALITY_WEIGHT * msoa_context["rurality_score"])
)

force_baseline_totals = (
    msoa_context
    .groupby("pfa_code", as_index=False)
    .agg(force_baseline_score=("baseline_score", "sum"))
)

msoa_context = msoa_context.merge(force_baseline_totals, on="pfa_code", how="left")
msoa_context["baseline_share"] = (
    msoa_context["baseline_score"] / msoa_context["force_baseline_score"]
)

msoa_context = msoa_context.merge(
    force_capacity_model[
        [
            "pfa_code",
            "pcso_fte",
            "regular_police_staff_fte",
            "total_capacity",
        ]
    ],
    on="pfa_code",
    how="left",
)

msoa_context["demand_reliability"] = 1.0
msoa_context.loc[
    msoa_context["pfa_code"].isin(BASELINE_ONLY_PFA_CODES),
    "demand_reliability",
] = 0.0

msoa_context["effective_rho"] = 1 - ((1 - RHO) * msoa_context["demand_reliability"])
msoa_context["capacity_base"] = msoa_context["effective_rho"] * msoa_context["total_capacity"]
msoa_context["capacity_var"] = (1 - msoa_context["effective_rho"]) * msoa_context["total_capacity"]
msoa_context["baseline_alloc_total"] = (
    msoa_context["capacity_base"] * msoa_context["baseline_share"]
)

force_allocation_settings = (
    msoa_context[
        [
            "pfa_code",
            "pfa_name",
            "total_capacity",
            "demand_reliability",
            "effective_rho",
            "capacity_base",
            "capacity_var",
        ]
    ]
    .drop_duplicates()
    .sort_values("pfa_name")
)

force_allocation_settings[force_allocation_settings["demand_reliability"] < 1]

fte_resources = (
    msoa_context
    .groupby(["pfa_code", "pfa_name"], as_index=False)
    .agg(
        msoa_count=("msoa_code", "nunique"),
        lsoa_count=("lsoa_count", "sum"),
        population=("population", "sum"),
        total_capacity=("total_capacity", "first"),
        pcso_fte=("pcso_fte", "first"),
        regular_police_staff_fte=("regular_police_staff_fte", "first"),
    )
)

fte_resources["fte_per_msoa"] = (
    fte_resources["total_capacity"] / fte_resources["msoa_count"]
)
fte_resources["msoas_per_100_fte"] = (
    fte_resources["msoa_count"] / fte_resources["total_capacity"] * 100
)
fte_resources["population_per_fte"] = (
    fte_resources["population"] / fte_resources["total_capacity"]
)

fte_resources = fte_resources.sort_values("pfa_name").reset_index(drop=True)


In [ ]:
months = pd.DataFrame({"month": sorted(msoa_bucket_demand["month"].unique())})

msoa_month_context = msoa_context.merge(months, how="cross")

baseline_allocation = msoa_month_context.merge(bucket_baseline_shares, how="cross")
baseline_allocation["baseline_alloc"] = (
    baseline_allocation["baseline_alloc_total"]
    * baseline_allocation["bucket_baseline_share"]
)

baseline_allocation = baseline_allocation[
    [
        "pfa_code",
        "pfa_name",
        "msoa_code",
        "msoa_name",
        "month",
        "bucket",
        "baseline_alloc",
        "total_capacity",
        "capacity_base",
        "capacity_var",
    ]
].copy()

baseline_allocation.head()

## Variable allocation

Variable allocation follows forecast demand within each force, month, and crime bucket.

In [ ]:
force_bucket_demand = (
    msoa_bucket_demand
    .groupby(["pfa_code", "pfa_name", "month", "bucket"], as_index=False)
    .agg(force_bucket_demand=("weighted_demand", "sum"))
)

force_month_demand = (
    force_bucket_demand
    .groupby(["pfa_code", "pfa_name", "month"], as_index=False)
    .agg(force_total_demand=("force_bucket_demand", "sum"))
)

force_bucket_demand = force_bucket_demand.merge(
    force_month_demand,
    on=["pfa_code", "pfa_name", "month"],
    how="left",
)

force_bucket_demand["bucket_demand_share"] = 0.0
positive_force_demand = force_bucket_demand["force_total_demand"] > 0
force_bucket_demand.loc[positive_force_demand, "bucket_demand_share"] = (
    force_bucket_demand.loc[positive_force_demand, "force_bucket_demand"]
    / force_bucket_demand.loc[positive_force_demand, "force_total_demand"]
)

force_bucket_demand = force_bucket_demand.merge(
    force_allocation_settings[
        [
            "pfa_code",
            "total_capacity",
            "demand_reliability",
            "effective_rho",
            "capacity_var",
        ]
    ],
    on="pfa_code",
    how="left",
)

force_bucket_demand["variable_bucket_capacity"] = (
    force_bucket_demand["capacity_var"]
    * force_bucket_demand["bucket_demand_share"]
)

force_bucket_demand.head()


In [ ]:
variable_allocation = msoa_bucket_demand.merge(
    force_bucket_demand[
        [
            "pfa_code",
            "pfa_name",
            "month",
            "bucket",
            "force_bucket_demand",
            "variable_bucket_capacity",
        ]
    ],
    on=["pfa_code", "pfa_name", "month", "bucket"],
    how="left",
)

variable_allocation["msoa_bucket_demand_share"] = 0.0
positive_bucket_demand = variable_allocation["force_bucket_demand"] > 0
variable_allocation.loc[positive_bucket_demand, "msoa_bucket_demand_share"] = (
    variable_allocation.loc[positive_bucket_demand, "weighted_demand"]
    / variable_allocation.loc[positive_bucket_demand, "force_bucket_demand"]
)

variable_allocation["variable_alloc"] = (
    variable_allocation["variable_bucket_capacity"]
    * variable_allocation["msoa_bucket_demand_share"]
)

variable_allocation = variable_allocation[
    [
        "pfa_code",
        "pfa_name",
        "msoa_code",
        "msoa_name",
        "month",
        "bucket",
        "demand_q25",
        "demand",
        "demand_q75",
        "weighted_demand_q25",
        "weighted_demand",
        "weighted_demand_q75",
        "force_bucket_demand",
        "variable_bucket_capacity",
        "msoa_bucket_demand_share",
        "variable_alloc",
    ]
].copy()

variable_allocation.head()


## MSOA allocation

In [ ]:
msoa_allocation = baseline_allocation.merge(
    variable_allocation[
        [
            "pfa_code",
            "msoa_code",
            "month",
            "bucket",
            "demand_q25",
            "demand",
            "demand_q75",
            "weighted_demand_q25",
            "weighted_demand",
            "weighted_demand_q75",
            "variable_alloc",
        ]
    ],
    on=["pfa_code", "msoa_code", "month", "bucket"],
    how="left",
)

for col in ["demand_q25", "demand", "demand_q75", "weighted_demand_q25", "weighted_demand", "weighted_demand_q75"]:
    msoa_allocation[col] = msoa_allocation[col].fillna(0)

msoa_allocation["variable_alloc"] = msoa_allocation["variable_alloc"].fillna(0)

msoa_allocation["total_alloc"] = (
    msoa_allocation["baseline_alloc"]
    + msoa_allocation["variable_alloc"]
)

msoa_allocation["capacity_source"] = capacity_source

msoa_allocation["share_of_force_capacity"] = (
    msoa_allocation["total_alloc"]
    / msoa_allocation["total_capacity"]
)

msoa_allocation = msoa_allocation[
    [
        "pfa_code",
        "pfa_name",
        "msoa_code",
        "msoa_name",
        "month",
        "bucket",
        "capacity_source",
        "demand_q25",
        "demand",
        "demand_q75",
        "weighted_demand_q25",
        "weighted_demand",
        "weighted_demand_q75",
        "baseline_alloc",
        "variable_alloc",
        "total_alloc",
        "share_of_force_capacity",
        "total_capacity",
    ]
].copy()

msoa_allocation.head()

In [ ]:
msoa_allocation = msoa_allocation.sort_values(
    ["pfa_code", "msoa_code", "bucket", "month"]
).copy()

msoa_allocation["prev_month_total_alloc"] = (
    msoa_allocation
    .groupby(["pfa_code", "msoa_code", "bucket"])["total_alloc"]
    .shift(1)
)

msoa_allocation["alloc_change_abs"] = (
    msoa_allocation["total_alloc"]
    - msoa_allocation["prev_month_total_alloc"]
)

msoa_allocation["alloc_change_pct"] = (
    msoa_allocation["alloc_change_abs"]
    / msoa_allocation["prev_month_total_alloc"]
    * 100
)

msoa_allocation["demand_iqr"] = (
    msoa_allocation["demand_q75"]
    - msoa_allocation["demand_q25"]
)

msoa_allocation["demand_iqr_ratio"] = (
    msoa_allocation["demand_iqr"]
    / msoa_allocation["demand"].clip(lower=1)
)

msoa_allocation["prev_month_demand"] = (
    msoa_allocation
    .groupby(["pfa_code", "msoa_code", "bucket"])["demand"]
    .shift(1)
)

msoa_allocation["demand_change_abs"] = (
    msoa_allocation["demand"]
    - msoa_allocation["prev_month_demand"]
)

msoa_allocation["demand_change_pct"] = (
    msoa_allocation["demand_change_abs"]
    / msoa_allocation["prev_month_demand"].clip(lower=1)
    * 100
)

wide_interval = msoa_allocation["demand_iqr_ratio"] >= UNCERTAINTY_IQR_RATIO_THRESHOLD
large_jump_pct = msoa_allocation["demand_change_pct"].abs() >= UNCERTAINTY_CHANGE_PCT_THRESHOLD
large_jump_abs = msoa_allocation["demand_change_abs"].abs() >= UNCERTAINTY_CHANGE_ABS_THRESHOLD

msoa_allocation["uncertainty_flag"] = (
    wide_interval
    & large_jump_pct
    & large_jump_abs
).fillna(False)

msoa_allocation.head()

## Save outputs

In [ ]:
final_output_path = DATA_DIR / "msoa_allocation.csv"
fte_resources_path = DATA_DIR / "fte_resources.csv"

msoa_allocation.to_csv(final_output_path, index=False)
fte_resources.to_csv(fte_resources_path, index=False)

with sqlite3.connect(ALLOC_DB_PATH) as conn:
    msoa_allocation.to_sql("msoa_allocation", conn, if_exists="replace", index=False)

print("Saved final allocation rows:", len(msoa_allocation))
print("Final output file:", final_output_path)
print("FTE resources file:", fte_resources_path)
print("Database:", ALLOC_DB_PATH)